# 📩 Spam Classifier using Logistic Regression + Optuna
We'll build a model to classify SMS messages as "spam" or "ham" (not spam), and optimize its performance using Optuna.

---

## 🔹 Step 1: Load Dataset
We are using the popular **SMS Spam Collection Dataset** from UCI. It contains 5,500+ labeled messages.

Columns:
- `label`: "ham" or "spam"
- `message`: the actual text message

Let's load and preview the data.


In [8]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

import optuna
# Load the dataset
df = pd.read_csv("/kaggle/input/sms-spam-collection-dataset/spam.csv", encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 🔹 Step 2: Preprocessing

We'll convert text to numeric using **TF-IDF vectorization**, and encode the labels into binary (0 = ham, 1 = spam).


In [9]:
# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])  # ham = 0, spam = 1

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(
    df['message'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english', lowercase=True, max_features=5000)


## 🔹 Step 3: Hyperparameter Optimization with Optuna

We'll tune:
- `C` (regularization strength)
- `penalty` (`l1` or `l2`)
- `solver` (based on penalty)

Objective: **maximize F1 Score** on the validation set.


In [11]:
def objective(trial):
    C = trial.suggest_loguniform('C', 1e-4, 10.0)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

    if penalty == 'l1':
        solver = 'liblinear'
    else:
        solver = trial.suggest_categorical('solver', ['liblinear', 'lbfgs', 'saga'])

    clf = Pipeline([
        ('tfidf', tfidf),
        ('lr', LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=1000))
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    return f1_score(y_val, y_pred)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)


[I 2025-07-24 10:53:07,767] A new study created in memory with name: no-name-8e7158f4-353c-4e62-b578-79ea09f62d62
/tmp/ipykernel_36/3603674272.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('C', 1e-4, 10.0)
[I 2025-07-24 10:53:07,850] Trial 0 finished with value: 0.0 and parameters: {'C': 0.01876925681560077, 'penalty': 'l1'}. Best is trial 0 with value: 0.0.
/tmp/ipykernel_36/3603674272.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('C', 1e-4, 10.0)
[I 2025-07-24 10:53:07,925] Trial 1 finished with value: 0.0 and parameters: {'C': 0.002928776621824908, 'penalty': 'l1'}. Best is trial 0 with value: 0.0.

## ✅ Step 4: Final Model Evaluation

We retrain a Logistic Regression model using the **best parameters from Optuna** and evaluate its F1 score.


In [12]:
print("Best trial:")
print(f"  F1 Score: {study.best_value:.4f}")
print(f"  Params: {study.best_params}")

# Build best model
best_params = study.best_params
penalty = best_params['penalty']
C = best_params['C']
solver = best_params.get('solver', 'liblinear')  # default for l1

final_model = Pipeline([
    ('tfidf', tfidf),
    ('lr', LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=1000))
])

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_val)

print(f"Final F1 Score on Validation Set: {f1_score(y_val, y_pred):.4f}")


Best trial:
  F1 Score: 0.9209
  Params: {'C': 9.86524532782154, 'penalty': 'l2', 'solver': 'liblinear'}
Final F1 Score on Validation Set: 0.9209
